In [1]:
###RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 7 PDF files to process

Processing: Cauvery-Water-Dispute.pdf


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


  ✓ Loaded 3 pages

Processing: Citizenship-Amendment-Act-2019.pdf
  ✓ Loaded 7 pages

Processing: Farmers (Empowerment and protection) bill, 2020.pdf
  ✓ Loaded 15 pages

Processing: India Farmer Demonstrations Continue Against Historic Agricultural Market Reforms_New Delhi_India_12-04-2020.pdf
  ✓ Loaded 4 pages

Processing: Language-dataset.pdf
  ✓ Loaded 15 pages

Processing: NewFarmActs2020.pdf
  ✓ Loaded 20 pages

Processing: The_Cauvery_River_Water_Dispute_A_Human_Rights_Per.pdf
  ✓ Loaded 7 pages

Total documents loaded: 71


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 71 documents into 396 chunks

Example chunk:
Content: Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the K...
Metadata: {'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [13]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    collection_name="rag_collection",
    embedding_function=embedding_function,
    persist_directory="./chroma_db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
import os
import warnings
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer

# Hide unnecessary warnings
warnings.filterwarnings("ignore")

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")

            # Load the embedding model
            self.model = SentenceTransformer(
                self.model_name,
                trust_remote_code=False
            )

            print("✅ Model loaded successfully!")
            print(f"Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model is not loaded.")

        print(f"Generating embeddings for {len(texts)} text(s)...")

        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        print(f"✅ Embeddings generated successfully!")
        print(f"Shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
Embedding Dimension: 384


In [16]:
chunks

[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20240406093916Z00'00'", 'title': 'Cauvery Water Dispute', 'author': 'Surbhi kataria', 'moddate': "D:20240406093916Z00'00'", 'source': '..\\data\\protests\\Cauvery-Water-Dispute.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Cauvery-Water-Dispute.pdf', 'file_type': 'pdf'}, page_content='Cauvery Water Dispute  The Kaveri is an interstate basin that originates in Karnataka and passes through Tamil Nadu and Pondicherry before draining into the Bay of Bengal. The total watershed of the Kaveri basin is 81,155 sq km, of which the river’s catchment area is about 34,273 sq km in Karnataka, 2,866 sq km in Kerala and the remaining 44,016 sq km in Tamil Nadu and Pondicherry. 1.The Harangi and Hemavati dams in Karnataka have been constructed on the Harangi and Hemavati rivers which are tributaries of the rivers Kaveri. The Krishna Raja Sagar Dam has b

In [18]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# (Optional) Generate embeddings just to inspect them
embeddings = embedding_manager.generate_embeddings(texts)

# Store documents in the vector database
vectorstore.add_documents(chunks)

Generating embeddings for 396 text(s)...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Embeddings generated successfully!
Shape: (396, 384)


['1097e45c-5bee-4f57-9e81-1d89aef08a3a',
 '9ceec8fb-337c-4108-a987-e6345fd6b965',
 '08afef84-d8e2-44a6-8924-460b830fa21a',
 '86b50bd3-df7e-4101-a2f6-3295850add3d',
 '29c63769-b563-4311-b7ae-ceb2bd9beee6',
 'f34abfd8-d613-47ba-8e1d-1a5e7cd8e4f0',
 'd92fcab5-3fde-452f-9ae6-73d6f6a36a9b',
 '48e5f756-b423-4f98-a911-cacd864b76ca',
 '483ed943-5690-4d7f-a96b-a06c619aa136',
 '383d4269-cb49-4bae-b7a2-d34de9e3201d',
 'c12eaf0a-9eb0-4139-b293-26363eb4b01f',
 'e951f770-0c73-4dcd-9b9a-800ff1aa342d',
 '3c76a282-86ba-44c5-88f4-a4467de6f984',
 '9cf662f7-91ae-4dbb-a315-18209ae462d9',
 '0eb32173-029d-4454-8681-ebe9d9123dff',
 '66e59e51-5441-42dc-89f7-1e680fb13098',
 'f96cce1d-7c52-4a40-b8c4-879962086a00',
 '8e3b3bca-5937-487f-b8b2-477acb087794',
 'a5643690-313d-4b0a-a7ce-a5777640000b',
 '8e06efa8-c5e3-47db-9238-1e80e146047b',
 '8e797fab-910b-4ad5-976c-98fa3f45702f',
 '780ff9c2-b283-4c11-99aa-126dc8cb56a7',
 'bdb6f90a-73df-45b2-8ba5-64202e3b482f',
 '388cb44d-3bb6-46f9-a9c6-665d9c0de6cb',
 'd6f62776-a9a9-

In [19]:
###Retriver pipeline from vector store

In [2]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            results = self.vector_store.similarity_search_with_score(
                query,
                k=top_k
            )

            retrieved_docs = []

            for i, (doc, score) in enumerate(results):

                retrieved_docs.append({
                    "id": i,
                    "content": doc.page_content,
                    "metadata": doc.metadata,
                    "similarity_score": score,
                    "rank": i + 1
                })

            print(f"Retrieved {len(retrieved_docs)} documents")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Create Retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

NameError: name 'vectorstore' is not defined

In [22]:
rag_retriever

In [23]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 text(s)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generated successfully!
Shape: (1, 384)
Error during retrieval: 'Chroma' object has no attribute 'collection'


[]

In [25]:
rag_retriever.retrieve("What are the various policies ")

Retrieving documents for query: 'What are the various policies '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 text(s)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generated successfully!
Shape: (1, 384)
Error during retrieval: 'Chroma' object has no attribute 'collection'


[]